In [1]:
from google.colab import drive
from pathlib import Path

MOUNT_POINT = "/content/ads_exp6_drive"

mount_path = Path(MOUNT_POINT)

# Mount only if not already mounted
if not (
    mount_path.exists()
    and any(mount_path.iterdir())
):
    drive.mount(MOUNT_POINT)

DRIVE_MOUNT = Path(MOUNT_POINT)

SHORTCUT_ID = (
    "1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL"
)

shortcut_project = (
    DRIVE_MOUNT
    / ".shortcut-targets-by-id"
    / SHORTCUT_ID
    / "Applied Data Science Project"
)

mydrive_project = (
    DRIVE_MOUNT
    / "MyDrive"
    / "Applied Data Science Project"
)

if shortcut_project.exists():

    PROJECT_ROOT = shortcut_project

elif mydrive_project.exists():

    PROJECT_ROOT = mydrive_project

else:

    raise FileNotFoundError(
        "Applied Data Science Project not found."
    )

print("✅ Project found:")
print(PROJECT_ROOT)

Mounted at /content/ads_exp6_drive
✅ Project found:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project


In [2]:
import joblib
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Experiment 6 directories
# ---------------------------------------------------------

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "exp4"
    / "best_loan_approval_model.joblib"
)

EXP6_DIR = (
    PROJECT_ROOT
    / "exp6_api"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "exp6"
)

EXP6_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------
# Check model
# ---------------------------------------------------------

print("=" * 70)
print("EXPERIMENT 6 SETUP")
print("=" * 70)

print(
    "Model exists:",
    MODEL_PATH.exists()
)

if not MODEL_PATH.exists():

    raise FileNotFoundError(
        f"Model not found: {MODEL_PATH}"
    )

# Load Experiment 4 best model
model = joblib.load(
    MODEL_PATH
)

print("\n✅ Model loaded successfully")

print(
    "Model type:",
    type(model).__name__
)

print("\nPipeline steps:")

if hasattr(model, "named_steps"):

    for name, step in model.named_steps.items():

        print(
            f"{name} -> "
            f"{type(step).__name__}"
        )

print("\nExperiment 6 API directory:")
print(EXP6_DIR)

print("\n✅ Ready to build FastAPI prediction API.")

EXPERIMENT 6 SETUP
Model exists: True


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid


✅ Model loaded successfully
Model type: Pipeline

Pipeline steps:
preprocessor -> ColumnTransformer
classifier -> HistGradientBoostingClassifier

Experiment 6 API directory:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp6_api

✅ Ready to build FastAPI prediction API.


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator _BinMapper from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator HistGradientBoostingClassifier from version 1.9.1 when using version 1.6.1. This might lead to breakin

In [4]:
import shutil

API_MODEL_PATH = (
    EXP6_DIR /
    "best_loan_approval_model.joblib"
)

shutil.copy2(
    MODEL_PATH,
    API_MODEL_PATH
)

print(" Model copied into API folder")

print("Source:")
print(MODEL_PATH)

print("\nAPI model:")
print(API_MODEL_PATH)

print(
    "\nExists:",
    API_MODEL_PATH.exists()
)

print(
    "Size:",
    round(
        API_MODEL_PATH.stat().st_size
        / (1024 * 1024),
        2
    ),
    "MB"
)

 Model copied into API folder
Source:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/models/exp4/best_loan_approval_model.joblib

API model:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp6_api/best_loan_approval_model.joblib

Exists: True
Size: 0.47 MB


In [5]:
APP_CODE = '''
from pathlib import Path

import joblib
import pandas as pd

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


# ---------------------------------------------------------
# Load trained model
# ---------------------------------------------------------

BASE_DIR = Path(__file__).resolve().parent

MODEL_PATH = (
    BASE_DIR /
    "best_loan_approval_model.joblib"
)

model = joblib.load(MODEL_PATH)


# ---------------------------------------------------------
# FastAPI application
# ---------------------------------------------------------

app = FastAPI(
    title="Loan Approval Prediction API",
    description=(
        "API for predicting loan application approval "
        "using the trained Experiment 4 machine learning model."
    ),
    version="1.0.0"
)


# ---------------------------------------------------------
# Input schema
# ---------------------------------------------------------

class LoanApplication(BaseModel):

    loan_amount: float = Field(
        ...,
        gt=0
    )

    risk_score: float | None = None

    dti: float | None = None

    employment_years: float | None = None

    application_year: int

    application_month: int = Field(
        ...,
        ge=1,
        le=12
    )

    application_quarter: int = Field(
        ...,
        ge=1,
        le=4
    )

    purpose: str

    state: str


# ---------------------------------------------------------
# Root endpoint
# ---------------------------------------------------------

@app.get("/")
def root():

    return {
        "message": "Loan Approval Prediction API",
        "status": "running"
    }


# ---------------------------------------------------------
# Health endpoint
# ---------------------------------------------------------

@app.get("/health")
def health():

    return {
        "status": "healthy",
        "model_loaded": True
    }


# ---------------------------------------------------------
# Prediction endpoint
# ---------------------------------------------------------

@app.post("/predict")
def predict(application: LoanApplication):

    try:

        input_df = pd.DataFrame([
            application.model_dump()
        ])

        prediction = int(
            model.predict(
                input_df
            )[0]
        )

        probability = float(
            model.predict_proba(
                input_df
            )[0][1]
        )

        decision = (
            "Accepted"
            if prediction == 1
            else "Rejected"
        )

        return {
            "prediction": prediction,
            "decision": decision,
            "approval_probability": round(
                probability,
                4
            )
        }

    except Exception as error:

        raise HTTPException(
            status_code=500,
            detail=str(error)
        )
'''

APP_PATH = (
    EXP6_DIR /
    "app.py"
)

APP_PATH.write_text(
    APP_CODE,
    encoding="utf-8"
)

print(" FastAPI app.py created:")
print(APP_PATH)

print("\nFile size:")
print(APP_PATH.stat().st_size, "bytes")

 FastAPI app.py created:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp6_api/app.py

File size:
2874 bytes


In [6]:
import sys
import importlib

from fastapi.testclient import TestClient

# Add API directory to Python path
if str(EXP6_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(EXP6_DIR)
    )

# Import FastAPI application
if "app" in sys.modules:
    del sys.modules["app"]

api_module = importlib.import_module(
    "app"
)

client = TestClient(
    api_module.app
)

# ---------------------------------------------------------
# Sample JSON request
# ---------------------------------------------------------

sample_request = {
    "loan_amount": 15000,
    "risk_score": 720,
    "dti": 18.5,
    "employment_years": 5,
    "application_year": 2018,
    "application_month": 6,
    "application_quarter": 2,
    "purpose": "debt_consolidation",
    "state": "CA"
}

response = client.post(
    "/predict",
    json=sample_request
)

print("=" * 70)
print("FASTAPI /predict TEST")
print("=" * 70)

print(
    "HTTP Status:",
    response.status_code
)

print("\nRequest JSON:")
print(sample_request)

print("\nResponse JSON:")
print(
    response.json()
)

if response.status_code == 200:

    print(
        "\n /predict endpoint working successfully"
    )

else:

    print(
        "\n API test failed"
    )

FASTAPI /predict TEST
HTTP Status: 200

Request JSON:
{'loan_amount': 15000, 'risk_score': 720, 'dti': 18.5, 'employment_years': 5, 'application_year': 2018, 'application_month': 6, 'application_quarter': 2, 'purpose': 'debt_consolidation', 'state': 'CA'}

Response JSON:
{'prediction': 1, 'decision': 'Accepted', 'approval_probability': 0.9968}

 /predict endpoint working successfully


In [7]:
REQUIREMENTS = """fastapi
uvicorn[standard]
pandas
numpy
joblib
scikit-learn==1.9.1
"""

REQUIREMENTS_PATH = (
    EXP6_DIR /
    "requirements.txt"
)

REQUIREMENTS_PATH.write_text(
    REQUIREMENTS,
    encoding="utf-8"
)

print(" requirements.txt created")
print(REQUIREMENTS_PATH)

print("\nContents:")
print(REQUIREMENTS)

 requirements.txt created
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp6_api/requirements.txt

Contents:
fastapi
uvicorn[standard]
pandas
numpy
joblib
scikit-learn==1.9.1



In [8]:
DOCKERFILE = """FROM python:3.13-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY best_loan_approval_model.joblib .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

DOCKERFILE_PATH = (
    EXP6_DIR /
    "Dockerfile"
)

DOCKERFILE_PATH.write_text(
    DOCKERFILE,
    encoding="utf-8"
)


DOCKERIGNORE = """__pycache__/
*.pyc
*.pyo
.ipynb_checkpoints/
.git/
.gitignore
"""

DOCKERIGNORE_PATH = (
    EXP6_DIR /
    ".dockerignore"
)

DOCKERIGNORE_PATH.write_text(
    DOCKERIGNORE,
    encoding="utf-8"
)

print(" Dockerfile created:")
print(DOCKERFILE_PATH)

print("\n .dockerignore created:")
print(DOCKERIGNORE_PATH)

print("\n===== DOCKERFILE =====")
print(DOCKERFILE)

 Dockerfile created:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp6_api/Dockerfile

 .dockerignore created:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp6_api/.dockerignore

===== DOCKERFILE =====
FROM python:3.13-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY best_loan_approval_model.joblib .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]



In [9]:
import json
from datetime import datetime

# ---------------------------------------------------------
# Save request
# ---------------------------------------------------------

REQUEST_PATH = (
    REPORT_DIR /
    "sample_prediction_request.json"
)

REQUEST_PATH.write_text(
    json.dumps(
        sample_request,
        indent=4
    ),
    encoding="utf-8"
)

# ---------------------------------------------------------
# Save response
# ---------------------------------------------------------

response_data = response.json()

RESPONSE_PATH = (
    REPORT_DIR /
    "sample_prediction_response.json"
)

RESPONSE_PATH.write_text(
    json.dumps(
        response_data,
        indent=4
    ),
    encoding="utf-8"
)

# ---------------------------------------------------------
# Save complete test evidence
# ---------------------------------------------------------

TEST_EVIDENCE = f"""
EXPERIMENT 6 - FASTAPI LOCAL TEST EVIDENCE
======================================================================

Endpoint:
POST /predict

HTTP Status:
{response.status_code}

Request JSON:
{json.dumps(sample_request, indent=4)}

Response JSON:
{json.dumps(response_data, indent=4)}

Result:
{'PASS - /predict endpoint is working successfully'
 if response.status_code == 200
 else 'FAIL'}

Test Method:
FastAPI TestClient

Model:
Experiment 4 HistGradientBoosting loan approval pipeline

Test Timestamp:
{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

TEST_EVIDENCE_PATH = (
    REPORT_DIR /
    "api_local_test_evidence.txt"
)

TEST_EVIDENCE_PATH.write_text(
    TEST_EVIDENCE,
    encoding="utf-8"
)

print(TEST_EVIDENCE)

print("\n Request saved:")
print(REQUEST_PATH)

print("\n Response saved:")
print(RESPONSE_PATH)

print("\n Test evidence saved:")
print(TEST_EVIDENCE_PATH)

print("\n===== EXPERIMENT 6 API FILES =====")

for path in EXP6_DIR.iterdir():
    print(
        " ",
        path.name,
        "-",
        round(
            path.stat().st_size / 1024,
            2
        ),
        "KB"
    )


EXPERIMENT 6 - FASTAPI LOCAL TEST EVIDENCE

Endpoint:
POST /predict

HTTP Status:
200

Request JSON:
{
    "loan_amount": 15000,
    "risk_score": 720,
    "dti": 18.5,
    "employment_years": 5,
    "application_year": 2018,
    "application_month": 6,
    "application_quarter": 2,
    "purpose": "debt_consolidation",
    "state": "CA"
}

Response JSON:
{
    "prediction": 1,
    "decision": "Accepted",
    "approval_probability": 0.9968
}

Result:
PASS - /predict endpoint is working successfully

Test Method:
FastAPI TestClient

Model:
Experiment 4 HistGradientBoosting loan approval pipeline

Test Timestamp:
2026-09-22 16:36:26


 Request saved:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/reports/exp6/sample_prediction_request.json

 Response saved:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/reports/exp6/sample_prediction_response.json

 Test evidenc

In [11]:
import subprocess
import time
import os

# Stop an old server if this cell was already run
if "uvicorn_process" in globals():
    try:
        uvicorn_process.terminate()
        uvicorn_process.wait(timeout=5)
    except:
        pass

# Start Uvicorn from the API directory
uvicorn_process = subprocess.Popen(
    [
        "python",
        "-m",
        "uvicorn",
        "app:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    cwd=str(EXP6_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Give server time to start
time.sleep(5)

print(" Uvicorn process started")
print("Process ID:", uvicorn_process.pid)

print("\nServer running:",
      uvicorn_process.poll() is None)

 Uvicorn process started
Process ID: 5472

Server running: True


In [12]:
import requests
import json

BASE_URL = "http://127.0.0.1:8000"

# ---------------------------------------------------------
# Health test
# ---------------------------------------------------------

health_response = requests.get(
    f"{BASE_URL}/health"
)

print("=" * 70)
print("HEALTH ENDPOINT TEST")
print("=" * 70)

print("Status:", health_response.status_code)
print("Response:", health_response.json())


# ---------------------------------------------------------
# Prediction test
# ---------------------------------------------------------

http_prediction_response = requests.post(
    f"{BASE_URL}/predict",
    json=sample_request
)

print("\n" + "=" * 70)
print("REAL HTTP /predict TEST")
print("=" * 70)

print(
    "HTTP Status:",
    http_prediction_response.status_code
)

print("\nRequest:")
print(
    json.dumps(
        sample_request,
        indent=4
    )
)

print("\nResponse:")
print(
    json.dumps(
        http_prediction_response.json(),
        indent=4
    )
)

if (
    health_response.status_code == 200
    and http_prediction_response.status_code == 200
):
    print(
        "\n FastAPI server successfully tested over HTTP"
    )
else:
    print(
        "\n HTTP API test failed"
    )

HEALTH ENDPOINT TEST
Status: 200
Response: {'status': 'healthy', 'model_loaded': True}

REAL HTTP /predict TEST
HTTP Status: 200

Request:
{
    "loan_amount": 15000,
    "risk_score": 720,
    "dti": 18.5,
    "employment_years": 5,
    "application_year": 2018,
    "application_month": 6,
    "application_quarter": 2,
    "purpose": "debt_consolidation",
    "state": "CA"
}

Response:
{
    "prediction": 1,
    "decision": "Accepted",
    "approval_probability": 0.9968
}

 FastAPI server successfully tested over HTTP


In [13]:
# ---------------------------------------------------------
# Save real HTTP test evidence
# ---------------------------------------------------------

HTTP_EVIDENCE = f"""
EXPERIMENT 6 - FASTAPI HTTP TEST EVIDENCE
======================================================================

SERVER:
Uvicorn

URL:
http://127.0.0.1:8000

HEALTH ENDPOINT
---------------
GET /health

HTTP Status:
{health_response.status_code}

Response:
{json.dumps(health_response.json(), indent=4)}


PREDICTION ENDPOINT
-------------------
POST /predict

HTTP Status:
{http_prediction_response.status_code}

Request:
{json.dumps(sample_request, indent=4)}

Response:
{json.dumps(http_prediction_response.json(), indent=4)}

Result:
{"PASS" if http_prediction_response.status_code == 200 else "FAIL"}
"""

HTTP_EVIDENCE_PATH = (
    REPORT_DIR /
    "http_api_test_evidence.txt"
)

HTTP_EVIDENCE_PATH.write_text(
    HTTP_EVIDENCE,
    encoding="utf-8"
)


# ---------------------------------------------------------
# Docker build/run commands
# ---------------------------------------------------------

DOCKER_COMMANDS = """EXPERIMENT 6 - DOCKER COMMANDS

Run these commands from the exp6_api folder.

1. Build Docker image

docker build -t loan-approval-api .

2. Run Docker container

docker run -d -p 8000:8000 --name loan-api loan-approval-api

3. Check running containers

docker ps

4. Test health endpoint

curl http://localhost:8000/health

5. Test prediction endpoint

curl -X POST http://localhost:8000/predict ^
  -H "Content-Type: application/json" ^
  -d "{\\"loan_amount\\":15000,\\"risk_score\\":720,\\"dti\\":18.5,\\"employment_years\\":5,\\"application_year\\":2018,\\"application_month\\":6,\\"application_quarter\\":2,\\"purpose\\":\\"debt_consolidation\\",\\"state\\":\\"CA\\"}"

6. Stop container

docker stop loan-api

7. Remove container

docker rm loan-api
"""

DOCKER_COMMANDS_PATH = (
    REPORT_DIR /
    "docker_commands.txt"
)

DOCKER_COMMANDS_PATH.write_text(
    DOCKER_COMMANDS,
    encoding="utf-8"
)

print(" HTTP test evidence saved:")
print(HTTP_EVIDENCE_PATH)

print("\n Docker commands saved:")
print(DOCKER_COMMANDS_PATH)

print("\n===== DOCKER COMMANDS =====")
print(DOCKER_COMMANDS)

 HTTP test evidence saved:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/reports/exp6/http_api_test_evidence.txt

 Docker commands saved:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/reports/exp6/docker_commands.txt

===== DOCKER COMMANDS =====
EXPERIMENT 6 - DOCKER COMMANDS

Run these commands from the exp6_api folder.

1. Build Docker image

docker build -t loan-approval-api .

2. Run Docker container

docker run -d -p 8000:8000 --name loan-api loan-approval-api

3. Check running containers

docker ps

4. Test health endpoint

curl http://localhost:8000/health

5. Test prediction endpoint

curl -X POST http://localhost:8000/predict ^
  -H "Content-Type: application/json" ^
  -d "{\"loan_amount\":15000,\"risk_score\":720,\"dti\":18.5,\"employment_years\":5,\"application_year\":2018,\"application_month\":6,\"application_quarter\":2,\"purpose\":\"debt_consol

In [15]:
import subprocess
import time

# Stop existing server
if "uvicorn_process" in globals():
    try:
        uvicorn_process.terminate()
        uvicorn_process.wait(timeout=5)
    except:
        pass

# Start server
uvicorn_process = subprocess.Popen(
    [
        "python",
        "-m",
        "uvicorn",
        "app:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ],
    cwd=str(EXP6_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print(
    " Server running:",
    uvicorn_process.poll() is None
)

 Server running: True


In [16]:
import requests

health = requests.get(
    "http://127.0.0.1:8000/health"
)

prediction = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample_request
)

print("===== HEALTH =====")
print("Status:", health.status_code)
print(health.json())

print("\n===== PREDICTION =====")
print("Status:", prediction.status_code)
print(prediction.json())

if health.status_code == 200 and prediction.status_code == 200:
    print("\n FASTAPI SERVER IS WORKING CORRECTLY")

===== HEALTH =====
Status: 200
{'status': 'healthy', 'model_loaded': True}

===== PREDICTION =====
Status: 200
{'prediction': 1, 'decision': 'Accepted', 'approval_probability': 0.9968}

 FASTAPI SERVER IS WORKING CORRECTLY


In [17]:
from google.colab import output

print("Opening FastAPI through the Colab port proxy...")

output.serve_kernel_port_as_window(
    8000
)

Opening FastAPI through the Colab port proxy...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [19]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8000,
    height=1000
)

print("Inside the displayed FastAPI page, open /docs")

<IPython.core.display.Javascript object>

Inside the displayed FastAPI page, open /docs


In [20]:
from pathlib import Path
import shutil

print("PROJECT ROOT:")
print(PROJECT_ROOT)

print("\n===== EXPERIMENT 6 API FILES =====")

if EXP6_DIR.exists():
    for p in EXP6_DIR.iterdir():
        print("", p.name)
else:
    print(" exp6_api folder missing")

print("\n===== EXPERIMENT 6 REPORT FILES =====")

if REPORT_DIR.exists():
    for p in REPORT_DIR.iterdir():
        print("", p.name)
else:
    print(" reports/exp6 folder missing")


# Target notebook location
EXP6_NOTEBOOK = (
    PROJECT_ROOT /
    "ADS_EXP_6.ipynb"
)

print("\nNotebook in project:")
print(EXP6_NOTEBOOK)

print(
    "Exists:",
    EXP6_NOTEBOOK.exists()
)

PROJECT ROOT:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project

===== EXPERIMENT 6 API FILES =====
 best_loan_approval_model.joblib
 app.py
 __pycache__
 requirements.txt
 Dockerfile
 .dockerignore

===== EXPERIMENT 6 REPORT FILES =====
 sample_prediction_request.json
 sample_prediction_response.json
 api_local_test_evidence.txt
 http_api_test_evidence.txt
 docker_commands.txt

Notebook in project:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/ADS_EXP_6.ipynb
Exists: False


In [22]:
import os
import subprocess
import base64
from getpass import getpass

# Make sure Git commands run inside your project
os.chdir(PROJECT_ROOT)

print("Current repository:")
print(os.getcwd())

print("\n===== BEFORE PUSH =====")
subprocess.run(
    ["git", "status"]
)

print("\nLatest commits:")
subprocess.run(
    ["git", "log", "--oneline", "-3"]
)

# Ask for GitHub token if not already available
if "github_token" not in globals():
    github_token = getpass(
        "Paste your GitHub token: "
    ).strip()

credentials = f"x-access-token:{github_token}"

encoded = base64.b64encode(
    credentials.encode("utf-8")
).decode("ascii")

auth_header = (
    f"AUTHORIZATION: basic {encoded}"
)

print("\nPushing Experiment 6 to GitHub...\n")

push = subprocess.run(
    [
        "git",
        "-c",
        f"http.extraHeader={auth_header}",
        "push",
        "origin",
        "main"
    ],
    capture_output=True,
    text=True
)

print(push.stdout)
print(push.stderr)

if push.returncode == 0:
    print(" EXPERIMENT 6 PUSHED TO GITHUB SUCCESSFULLY")
else:
    print(" GitHub push failed")

Current repository:
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project

===== BEFORE PUSH =====

Latest commits:
Paste your GitHub token: ··········

Pushing Experiment 6 to GitHub...


Everything up-to-date

 EXPERIMENT 6 PUSHED TO GITHUB SUCCESSFULLY


In [24]:
import os
from pathlib import Path

print("PROJECT_ROOT:", PROJECT_ROOT)

EXP6_NOTEBOOK = PROJECT_ROOT / "ADS_EXP_6.ipynb"

print("\n===== CHECK =====")
print("Notebook:", EXP6_NOTEBOOK.exists())
print("API folder:", EXP6_DIR.exists())
print("Reports folder:", REPORT_DIR.exists())

print("\n===== API FILES =====")
if EXP6_DIR.exists():
    for p in EXP6_DIR.iterdir():
        print("✅", p.name)

print("\n===== REPORT FILES =====")
if REPORT_DIR.exists():
    for p in REPORT_DIR.iterdir():
        print("✅", p.name)

PROJECT_ROOT: /content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project

===== CHECK =====
Notebook: False
API folder: True
Reports folder: True

===== API FILES =====
✅ best_loan_approval_model.joblib
✅ app.py
✅ __pycache__
✅ requirements.txt
✅ Dockerfile
✅ .dockerignore

===== REPORT FILES =====
✅ sample_prediction_request.json
✅ sample_prediction_response.json
✅ api_local_test_evidence.txt
✅ http_api_test_evidence.txt
✅ docker_commands.txt


In [26]:
from pathlib import Path
import shutil

EXP6_NOTEBOOK = (
    PROJECT_ROOT /
    "ADS_EXP_6.ipynb"
)

possible_roots = [
    Path("/content/ads_exp6_drive"),
    Path("/content/drive"),
]

matches = []

for root in possible_roots:
    if root.exists():
        matches.extend(
            [
                p for p in root.rglob("*.ipynb")
                if p.name.lower() == "ads_exp_6.ipynb"
            ]
        )

print("Found copies:", len(matches))

for i, path in enumerate(matches, 1):
    print(i, path)

if matches:

    source = matches[0]

    print("\nUsing:")
    print(source)

    if source.resolve() != EXP6_NOTEBOOK.resolve():
        shutil.copy2(
            source,
            EXP6_NOTEBOOK
        )

    print(
        "\n Final notebook exists:",
        EXP6_NOTEBOOK.exists()
    )

else:

    print(
        "\n ADS_EXP_6.ipynb not found."
    )

Found copies: 2
1 /content/ads_exp6_drive/MyDrive/Colab Notebooks/ADS_EXP_6.ipynb
2 /content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/FINAL/ADS_EXP_6.ipynb

Using:
/content/ads_exp6_drive/MyDrive/Colab Notebooks/ADS_EXP_6.ipynb

✅ Final notebook exists: True


In [28]:
import os
import subprocess

os.chdir(PROJECT_ROOT)

print("=" * 70)
print("EXPERIMENT 6 FINAL FILE CHECK")
print("=" * 70)

checks = {
    "Notebook":
        PROJECT_ROOT / "ADS_EXP_6.ipynb",

    "FastAPI app":
        EXP6_DIR / "app.py",

    "Dockerfile":
        EXP6_DIR / "Dockerfile",

    "Requirements":
        EXP6_DIR / "requirements.txt",

    "Model":
        EXP6_DIR / "best_loan_approval_model.joblib",

    "Docker test script":
        EXP6_DIR / "docker_test.ps1",

    "README":
        EXP6_DIR / "README.md",

    "API test evidence":
        REPORT_DIR / "api_local_test_evidence.txt",

    "HTTP test evidence":
        REPORT_DIR / "http_api_test_evidence.txt",

    "Docker commands":
        REPORT_DIR / "docker_commands.txt"
}

all_ok = True

for name, path in checks.items():

    exists = path.exists()

    if not exists:
        all_ok = False

    print(
        f"{'' if exists else ''} {name}"
    )

print("=" * 70)

if all_ok:
    print(" EXPERIMENT 6 FILES READY FOR GIT")
else:
    print(" Some Experiment 6 files are missing")

EXPERIMENT 6 FINAL FILE CHECK
 Notebook
 FastAPI app
 Dockerfile
 Requirements
 Model
 Docker test script
 README
 API test evidence
 HTTP test evidence
 Docker commands
 Some Experiment 6 files are missing


In [29]:
checks = {
    "Notebook":
        PROJECT_ROOT / "ADS_EXP_6.ipynb",

    "FastAPI app":
        EXP6_DIR / "app.py",

    "Dockerfile":
        EXP6_DIR / "Dockerfile",

    "Docker Ignore":
        EXP6_DIR / ".dockerignore",

    "Requirements":
        EXP6_DIR / "requirements.txt",

    "Model":
        EXP6_DIR / "best_loan_approval_model.joblib",

    "Docker test script":
        EXP6_DIR / "docker_test.ps1",

    "README":
        EXP6_DIR / "README.md",

    "API test evidence":
        REPORT_DIR / "api_local_test_evidence.txt",

    "HTTP test evidence":
        REPORT_DIR / "http_api_test_evidence.txt",

    "Sample request":
        REPORT_DIR / "sample_prediction_request.json",

    "Sample response":
        REPORT_DIR / "sample_prediction_response.json",

    "Docker commands":
        REPORT_DIR / "docker_commands.txt"
}

print("=" * 75)
print("EXPERIMENT 6 DETAILED FILE CHECK")
print("=" * 75)

missing_files = []

for name, path in checks.items():

    if path.exists():
        status = "PRESENT"
    else:
        status = "MISSING"
        missing_files.append((name, path))

    print(f"{status:8} | {name}")
    print(f"         | {path}")

print("=" * 75)

print("\nMissing count:", len(missing_files))

for name, path in missing_files:
    print("MISSING:", name)

EXPERIMENT 6 DETAILED FILE CHECK
PRESENT  | Notebook
         | /content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/ADS_EXP_6.ipynb
PRESENT  | FastAPI app
         | /content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp6_api/app.py
PRESENT  | Dockerfile
         | /content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp6_api/Dockerfile
PRESENT  | Docker Ignore
         | /content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp6_api/.dockerignore
PRESENT  | Requirements
         | /content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp6_api/requirements.txt
PRESENT  | Model
         | /content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp

In [31]:
POWERSHELL_SCRIPT = r'''
Write-Host "=============================================="
Write-Host "ADS EXPERIMENT 6 - DOCKER LOCAL TEST"
Write-Host "=============================================="

Write-Host "`n[1] Building Docker image..."
docker build -t loan-approval-api .

if ($LASTEXITCODE -ne 0) {
    Write-Host "DOCKER BUILD FAILED"
    exit 1
}

docker rm -f loan-api 2>$null

Write-Host "`n[2] Starting Docker container..."
docker run -d -p 8000:8000 --name loan-api loan-approval-api

if ($LASTEXITCODE -ne 0) {
    Write-Host "DOCKER RUN FAILED"
    exit 1
}

Write-Host "`nWaiting for API..."
Start-Sleep -Seconds 8

Write-Host "`n[3] Running container:"
docker ps

Write-Host "`n[4] Testing /health..."

$health = Invoke-RestMethod `
    -Uri "http://localhost:8000/health" `
    -Method GET

$health | ConvertTo-Json

Write-Host "`n[5] Testing /predict..."

$body = @{
    loan_amount = 15000
    risk_score = 720
    dti = 18.5
    employment_years = 5
    application_year = 2018
    application_month = 6
    application_quarter = 2
    purpose = "debt_consolidation"
    state = "CA"
} | ConvertTo-Json

$prediction = Invoke-RestMethod `
    -Uri "http://localhost:8000/predict" `
    -Method POST `
    -ContentType "application/json" `
    -Body $body

$prediction | ConvertTo-Json

Write-Host "`n=============================================="
Write-Host "DOCKER CONTAINER TEST PASSED"
Write-Host "=============================================="

Write-Host "`nStopping container..."
docker stop loan-api
docker rm loan-api
'''

docker_test_path = EXP6_DIR / "docker_test.ps1"

docker_test_path.write_text(
    POWERSHELL_SCRIPT,
    encoding="utf-8"
)

print("docker_test.ps1 created")
print(docker_test_path)

docker_test.ps1 created
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp6_api/docker_test.ps1


In [32]:
README_LINES = [
    "# Experiment 6 - Containerization & API Deployment",
    "",
    "## Aim",
    "Containerization and API Deployment.",
    "",
    "## Objective",
    "Package the trained loan approval model in Docker and expose predictions through FastAPI.",
    "",
    "## API Endpoints",
    "",
    "### GET /",
    "Checks whether the API is running.",
    "",
    "### GET /health",
    "Checks API and model availability.",
    "",
    "### POST /predict",
    "Predicts whether a loan application is Accepted or Rejected.",
    "",
    "## Example Request",
    "",
    "{",
    '  "loan_amount": 15000,',
    '  "risk_score": 720,',
    '  "dti": 18.5,',
    '  "employment_years": 5,',
    '  "application_year": 2018,',
    '  "application_month": 6,',
    '  "application_quarter": 2,',
    '  "purpose": "debt_consolidation",',
    '  "state": "CA"',
    "}",
    "",
    "## Docker Build",
    "",
    "docker build -t loan-approval-api .",
    "",
    "## Docker Run",
    "",
    "docker run -d -p 8000:8000 --name loan-api loan-approval-api",
    "",
    "## Health Endpoint",
    "",
    "http://localhost:8000/health",
    "",
    "## Swagger Documentation",
    "",
    "http://localhost:8000/docs",
    "",
    "## Automated Windows Test",
    "",
    "powershell -ExecutionPolicy Bypass -File docker_test.ps1"
]

README_TEXT = "\n".join(README_LINES)

readme_path = EXP6_DIR / "README.md"

readme_path.write_text(
    README_TEXT,
    encoding="utf-8"
)

print(" README.md created")
print(readme_path)

 README.md created
/content/ads_exp6_drive/.shortcut-targets-by-id/1Qa-0wb-sSO4lC_krs-UYMHB3HiVqPSjL/Applied Data Science Project/exp6_api/README.md


In [33]:
checks = {
    "Notebook":
        PROJECT_ROOT / "ADS_EXP_6.ipynb",

    "FastAPI app":
        EXP6_DIR / "app.py",

    "Dockerfile":
        EXP6_DIR / "Dockerfile",

    "Docker Ignore":
        EXP6_DIR / ".dockerignore",

    "Requirements":
        EXP6_DIR / "requirements.txt",

    "Model":
        EXP6_DIR / "best_loan_approval_model.joblib",

    "Docker test script":
        EXP6_DIR / "docker_test.ps1",

    "README":
        EXP6_DIR / "README.md",

    "API test evidence":
        REPORT_DIR / "api_local_test_evidence.txt",

    "HTTP test evidence":
        REPORT_DIR / "http_api_test_evidence.txt",

    "Sample request":
        REPORT_DIR / "sample_prediction_request.json",

    "Sample response":
        REPORT_DIR / "sample_prediction_response.json",

    "Docker commands":
        REPORT_DIR / "docker_commands.txt"
}

print("=" * 72)
print("EXPERIMENT 6 PRE-GIT CHECK")
print("=" * 72)

all_ready = True

for name, path in checks.items():
    exists = path.exists()

    if not exists:
        all_ready = False

    print(
        f"{'' if exists else ''} {name}"
    )

print("=" * 72)

if all_ready:
    print(" ALL EXPERIMENT 6 FILES ARE READY FOR GIT")
else:
    print(" SOME FILES ARE STILL MISSING")

EXPERIMENT 6 PRE-GIT CHECK
 Notebook
 FastAPI app
 Dockerfile
 Docker Ignore
 Requirements
 Model
 Docker test script
 README
 API test evidence
 HTTP test evidence
 Sample request
 Sample response
 Docker commands
 ALL EXPERIMENT 6 FILES ARE READY FOR GIT
